In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pyspark.sql import functions as F
import yaml

In [0]:
PNG_DIR = "/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_plots"
OUT_CSV = "/Volumes/bmqg/default_bronze/fatemeh/final_project/allelic_effect_linearity.csv"


from pyspark.sql import functions as F
CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

PATHS = CONFIG["paths"]
DATA  = CONFIG["data"]

GWAS_TABLE   = DATA["gwas_table"]
TAGLO_TABLE  = PATHS["TAGLO_TABLE"]
PHENO_PATH   = PATHS["aroma_matrix"]

print("GWAS table :", GWAS_TABLE)
print("TAGLO table:", TAGLO_TABLE)
print("Pheno path :", PHENO_PATH)


GWAS table : bmqg.gwas.run_local_20251207
TAGLO table: bmqg.default_silver.taglotype_silver
Pheno path : /Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/aroma_matrix_GWAS_clean.csv


In [0]:
# ---- Load phenotype table into pandas (this will be used by the dashboard)
pheno_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PHENO_PATH)
    .toPandas()
)

print("pheno_df shape:", pheno_df.shape)
print("pheno_df columns sample:", list(pheno_df.columns)[:10])


pheno_df shape: (94, 68)
pheno_df columns sample: ['Variety', '(E)-2-Decenal', '(E)-2-Heptenal', '(E, E)-2,4-Decadienal', '(E, E)-3,5-Octadien-2-one', '1-Heptanol', '1-Hexanol, 2-ethyl-', '1-Nonanol', '1-Octen-3-ol', '1-Octyn-3-ol']


In [0]:
res = (
    spark.table(GWAS_TABLE)
    .filter(F.col("p_wald") < 1e-6)
    .select(
        "trait",
        F.explode(
            F.array("taglo_id1","taglo_id2","taglo_id3","taglo_id4")
        ).alias("taglo_id")
    )
    .filter(F.col("taglo_id").isNotNull())
    .filter(F.col("taglo_id") > 0)
    .groupBy("trait")
    .agg(F.collect_set("taglo_id").alias("taglo_ids"))
    .toPandas()
)

print("Traits available:", len(res))
display(res.head())


Traits available: 59


,trait,taglo_ids
0,1-Phenylethanol,"[131677, 131837, 257998, 131737, 131734, 13152..."
1,Propanal,"[159916, 159584, 260621, 160349, 159965, 15987..."
2,"1-Hexanol, 2-ethyl-","[554308, 556153, 118734, 545024, 217029, 25128..."
3,2-Nonanone,"[47107, 388629, 457779, 376015, 49534, 447386,..."
4,Tridecanal,"[219792, 219789, 258132, 494675, 494677, 21971..."


In [0]:
def build_dosage_df_from_taglos(spark, taglo_table, taglo_ids, pheno_df, trait):
    from pyspark.sql import functions as F

    gt = (
        spark.table(taglo_table)
        .filter(F.col("taglo_id").isin([int(t) for t in taglo_ids]))
        .select(
            F.lower(F.trim(F.col("variety"))).alias("Variety"),
            F.col("value").cast("int").alias("dosage")
        )
        .toPandas()
    )

    if gt.empty:
        return None

    gt = gt[gt["dosage"].isin([0, 1, 2, 3, 4])]

    if trait not in pheno_df.columns:
        raise KeyError(f"Trait '{trait}' not found in pheno_df columns.")

    ph = pheno_df[["Variety", trait]].copy()
    ph["Variety"] = ph["Variety"].astype(str).str.lower().str.strip()
    ph = ph.rename(columns={trait: "trait_value"})

    df = gt.merge(ph, on="Variety", how="inner")
    return None if df.empty else df


In [0]:
def manual_dosage_filter(df, remove_dosages):
    return df[~df["dosage"].isin(set(remove_dosages))].copy()


In [0]:
def compute_effect_and_fc(df, baseline=0, stat="median"):
    g = df.groupby("dosage")["trait_value"]

    if baseline not in g.groups or g.ngroups < 2:
        return None, None

    base = getattr(g.get_group(baseline), stat)()

    effects = {
        d: getattr(v, stat)() - base
        for d, v in g
    }

    max_d = max(effects, key=lambda k: abs(effects[k]))
    fc = getattr(g.get_group(max_d), stat)() / base if base != 0 else np.nan

    return effects, fc


In [0]:
def run_trait_dashboard(trait, remove_dosages, plot_type):
    clear_output(wait=True)

    row = res[res["trait"] == trait].iloc[0]
    taglo_ids = row["taglo_ids"]

    print(f"Trait: {trait}")
    print(f"Number of taglo_ids: {len(taglo_ids)}")
    print("Taglo IDs:", sorted(taglo_ids))

    df = build_dosage_df_from_taglos(
        spark,
        TAGLO_TABLE,
        taglo_ids,
        pheno_df,   #  FIX: use pheno_df, not ph
        trait
    )

    if df is None:
        print("No data available")
        return

    summary = (
        df.groupby("dosage")["trait_value"]
        .agg(count="count", median="median", mean="mean")
        .reset_index()
        .sort_values("dosage")
    )

    df_filt = manual_dosage_filter(df, remove_dosages)
    effect, fc = compute_effect_and_fc(df_filt)

    print("\nRemoved dosages:", list(remove_dosages))
    print("Effect (Δ vs baseline):", effect)
    print("Fold change:", fc)

    display(summary)

    plt.figure(figsize=(6,4))
    if plot_type == "box":
        df_filt.boxplot(column="trait_value", by="dosage")
    else:
        data = [
            df_filt[df_filt["dosage"] == d]["trait_value"]
            for d in sorted(df_filt["dosage"].unique())
        ]
        plt.violinplot(data, positions=sorted(df_filt["dosage"].unique()), showmedians=True)
        plt.xticks(sorted(df_filt["dosage"].unique()))

    plt.title(f"{trait} (after manual dosage removal)")
    plt.suptitle("")
    plt.xlabel("Allelic dosage")
    plt.ylabel(trait)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [0]:
trait_dd = widgets.Dropdown(
    options=sorted(res["trait"].tolist()),
    description="Trait:"
)

dosage_ms = widgets.SelectMultiple(
    options=[0,1,2,3,4],
    description="Remove dosage",
    layout=widgets.Layout(width="260px", height="130px")
)

plot_tb = widgets.ToggleButtons(
    options=["box", "violin"],
    description="Plot:"
)

widgets.interact(
    run_trait_dashboard,
    trait=trait_dd,
    remove_dosages=dosage_ms,
    plot_type=plot_tb
);


interactive(children=(Dropdown(description='Trait:', options=('(E)-2-Decenal', '(E)-2-Heptenal', '(E, E)-3,5-O…